<img src = "https://drive.google.com/uc?export=view&id=1Hh_G3M13P9xSNgSiQ-WnALg93XwK_hG8" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 5 - *Modelos generativos***
---

En esta tarea utilizarás Transformers para convertir texto natural en representaciones mediante emojis. El ejercicio permite comprender cómo los modelos generativos capturan la semántica del texto y su contexto emocional.

> **Importante: Recomendamos utilizar GPU para la ejecución de este notebook, ya que puede tomar mucho tiempo la ejeución de algunos casos de prueba en caso de que no se utilice. Tenga en cuenta que, con el propósito de guardar recursos, los primeros puntos se pueden solucionar sin utilizarlo**

**Descripción del problema**
---
La representación de texto mediante emojis trasciende la traducción directa palabra-a-símbolo. Requiere que el modelo:
- Comprenda el contexto semántico del texto de entrada
- Identifique conceptos clave y emociones implícitas
- Genere secuencias coherentes de emojis que preserven el significado original
- Mantenga la fluidez en la representación visual

In [ ]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

In [ ]:
import rlxcrypt
import session

grader = session.LoginSequence("DL-GroupMLDS-5-2026-2@8c487bd7-4c95-4b75-88d5-9e7093aae729")

> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

In [ ]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
import pandas as pd
import re
import json


%matplotlib inline
plt.style.use("ggplot")
# Seleccionamos las semillas para efectos de reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
np.random.seed(0)

In [ ]:
# Versiones de las librerías usadas.
!python --version
print('NumPy', np.__version__)
print('keras', keras.__version__)

Esta actividad se realizó con las siguientes versiones:
* Python 3.11.12
* NumPy 2.0.2
* keras 3.8.0

## **Cargar Datos**
---

Vamos cargar nuestro conjunto de datos. Este posee dos columnas: la primera contiene 10,000 textos en inglés  y en la segunda su representación por medio de emojis. Si bien esta puede parecer una tarea de traducción simple, el uso de emojis como concepto le exige al modelo la capacidad de abstraer patrones complejos.

In [ ]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1HjGoRLLnLdEWjB1peXGFQXNGvbPRaAlj

In [ ]:
with open("/content/emoji_translations.json", 'r', encoding='utf-8') as f:
    data_dict = json.load(f)

data = pd.DataFrame(data_dict.items(), columns=['ingles', 'emojis'])
data = data.dropna()

train_texts = data.loc[:int(len(data)*0.7)]
test_texts = data.loc[int(len(data)*0.7):]

print(f'Numero de textos de entrenamiento: {len(train_texts)}')
print(f'Numero de textos de prueba: {len(test_texts)}')

veamos algunos ejemplos:

In [ ]:
train_texts.head()

# **1. Preprocesamiento**
---
En este primer punto deberá implementar una función de prerpocesamiento de texto adaptada al problema presentado. Para eso deberá:


**Para la columna de texto:**

* Convertir todo el texto a minúsculas
* Eliminar caracteres especiales (mantener solo letras y espacios)
* Eliminar espacios repetidos
* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Para la columna de emojis:**

* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Entradas**:

* **`data`**: `pd.DataFrame`,  DataFrame que contiene las columnas 'texto' y 'emojis'

**Salida**:

* **`text_out`**: `pd.DataFrame`, DataFrame con el texto preprocesado según las reglas anteriores

In [ ]:
# FUNCIÓN CALIFICADA preprocess
def preprocess(data):
    def preprocess_text(text):
        pass
    def preprocess_emoji(emoji):
        pass

    text_out = data.copy()
    text_out['ingles'] = text_out['ingles'].map(preprocess_text)
    text_out['emojis'] = text_out['emojis'].map(preprocess_emoji)
    return text_out

In [ ]:
# TEST CELL
train_texts_preprocess = preprocess(train_texts.copy())
test_texts_preprocess = preprocess(test_texts.copy())


print('Texto en inglés')
print(train_texts_preprocess.iloc[0]['ingles'])
print('Emojis')
print(train_texts_preprocess.iloc[0]['emojis'])

**Salida esperada:**
```
Texto en ingles
[BOS] the ratification and implementation of the updated ilo conventions vote [EOS]
Emojis
[BOS] ✅ 🗳 ️ [EOS]
```




## **Evaluar código**

In [ ]:
grader.run_test("Test 1_1", globals())

In [ ]:
grader.run_test("Test 1_2", globals())

# **2. Vectorización**
----
Cree y ajuste una capa de vectorización para cada tipo de secuencia (texto y emojis).

**Entrada**:
* **`X`**:  `pd.DataFrame`, DataFrame que contiene el texto normalizado en `ingles` y `emoji`.
* **`seq_len`**: Tamaño máximo de la secuencia.
* **`vocab_size`**: Tamaño máximo del vocabulario.

**Salida**:
* **`en_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma inglés.

* **`em_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma emojis.


> **Nota**:Tenga presente que en el caso de la columna emojis su función solo debera añadir los tokens de incio y de parada, mientras que el texto debe ser preprocesado.

In [ ]:
# FUNCIÓN CALIFICADA vectorize
def vectorize(X, seq_len, vocab_size):
    en_vectorizer = None
    em_vectorizer = None
    return en_vectorizer, em_vectorizer

In [ ]:
# TEST_CELL
en_vectorizer, emoji_vectorizer = vectorize(train_texts_preprocess, 25, 5000)
train_en_vec, train_emoji_vec = en_vectorizer(train_texts_preprocess['ingles']).numpy(), emoji_vectorizer(train_texts_preprocess['emojis']).numpy()
test_en_vec, test_emoji_vec = en_vectorizer(test_texts_preprocess['ingles']).numpy(), emoji_vectorizer(test_texts_preprocess['emojis']).numpy()

print(f'Tamaño de las entradas: {train_en_vec.shape}')
print(f'Tamaño de las salidas: {train_emoji_vec.shape}')

**Salida esperada**:
```
Tamaño de las entradas: (6605, 26)
Tamaño de las salidas: (6605, 26)

```

## **Evaluar código**

In [ ]:
grader.run_test("Test 2_1", globals())

In [ ]:
grader.run_test("Test 2_2", globals())

# **3. Codificador**
----
Implemente una capa personalizada de Keras que represente el codificador de un Transformer.


**Entrada**:
* **`vocab_size`**: `int`, Tamaño del vocabulario de entrada
* **`maxlen`**: `int`, Longitud máxima de la secuencia de entrada
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en el mecanismo de auto-atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

In [ ]:
# FUNCIÓN CALIFICADA get_EncoderLayer
def get_EncoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class encoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            pass

        def call(self, inputs):
            pass

    layer = None
    return layer

In [ ]:
# TEST_CELL
encoder_layer = get_EncoderLayer(5000, 25, 128, 2, 32, .1)
encoded_x = encoder_layer(train_en_vec[:10])
print(f'Codificación de sequencias de entrenamiento: {encoded_x.shape}')

**Salida esperada**
```
Codificación de sequencias de entrenamiento: (10, 26, 128)
```

## **Evaluar código**

In [ ]:
grader.run_test("Test 3_1", globals())

In [ ]:
grader.run_test("Test 3_2", globals())

# **4. Decodificador**
----
Implemente una capa personalizada de Keras que represente el decodificador de un Transformer.

**Entrada**:

* **`vocab_size`**: `int`, Tamaño del vocabulario de salida
* **`maxlen`**: `int`, Longitud máxima de la secuencia de salida  
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en los mecanismos de atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

In [ ]:
# FUNCIÓN CALIFICADA get_DecoderLayer
def get_DecoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class Decoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            pass

        def call(self, inputs, context_tokens):
            pass

    layer = Decoder(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate)
    return layer

In [ ]:
# TEST_CELL
decoder_layer = get_DecoderLayer(5000, 25, 128, 2, 32, .1)
decoded_x = decoder_layer(train_emoji_vec[:10,:-1],encoded_x)
print(f'Decodificación de sequencias de entrenamiento: {decoded_x.shape}')

## **Evaluar código**

In [ ]:
grader.run_test("Test 4_1", globals())

In [ ]:
grader.run_test("Test 4_2", globals())

# **5. Construir modelo**
----
Implemente la función `get_model` que construye el modelo Transformer completo combinando el codificador y decodificador. El modelo debe realizar la tarea de secuencia-a-secuencia para convertir texto en inglés a representaciones con emojis.

**Entrada**:

* **`max_len`**: `int`, Longitud máxima de las secuencias
* **`vocab_size`**: `int`, Tamaño del vocabulario (debe ser el mismo para ambos vocabularios)
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas de atención
* **`ff_dim`**: `int`, Dimensión de la capa feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`model`**: `keras.Model`, Modelo Transformer encoder-decoder completo

> **Nota**: El modelo implementa la arquitectura encoder-decoder típica para tareas de traducción/generación de secuencias, donde el decodificador utiliza tanto su entrada desplazada como el contexto del codificador.

In [ ]:
# FUNCIÓN CALIFICADA get_model
def get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate):
    # Fijamos una semilla para efectos de reproducibiidad
    np.random.seed(0)
    keras.utils.set_random_seed(0)

    keras.backend.clear_session()


    model = None
    return model


In [ ]:
# TEST_CELL
max_len = 25
vocab_size = 5000
embed_dim = 128
num_heads = 2
ff_dim = 32
rate = 0.01

model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate)

model.summary()

**Salida esperada:**
```
Model: "functional_2"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (encoder)   │ (None, 26, 128)   │    784,160 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 25, 5000)  │  1,561,256 │ input_layer_1[0]… │
│                     │                   │            │ encoder[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘
 Total params: 2,345,416 (8.95 MB)
 Trainable params: 2,345,416 (8.95 MB)
 Non-trainable params: 0 (0.00 B)
```

## **Evaluar código**

In [ ]:
grader.run_test("Test 5_1", globals())

In [ ]:
grader.run_test("Test 5_2", globals())

# **6. Entrenar el modelo**
----
Complete la función `train_model` que recibe un modelo y lo ajusta usando las secuencias de entrenamiento.

**Entrada**:

* **`model`**: `keras.Model`, modelo de _Keras_ compilado.
* **`train_text`**:  `array`, Arreglo con los textos vectorizados de entrenamiento en inglés.
* **`train_emoji`**:  `array`, Arreglo con los textos vectorizados de entrenamiento de emojis.
* **`test_text`**:  `array`, Arreglo con los textos vectorizados de prueba en inglés.
* **`test_emoji`**:  `array`, Arreglo con los textos vectorizados de prueba en emojis.
* **`learning_rate`**: `float`, tasa de aprendizaje.
* **`epochs`**: `int`, Número de epocas.
* **`batch_size`**: `int`, tamaño del lote.

**Salida**:
* **`history`**: Objeto, objeto que contiene las métricas de evaluación y entrenamiento a lo largo de las épocas.

> **Nota**: Recuerde desfasar por un token la salida esperada de la entrada.



In [ ]:
def causal_loss(y_true,y_pred):
    masc = keras.ops.cast(y_true != 0, dtype="float32")
    y_true = keras.ops.one_hot(y_true, 5000, axis=-1)

    loss_fn = keras.losses.CategoricalCrossentropy(from_logits=False, reduction='none')
    loss = loss_fn(y_true, y_pred)
    loss = loss * masc
    loss = keras.ops.sum(loss)/keras.ops.sum(masc)
    return loss

In [ ]:
# FUNCIÓN CALIFICADA train_model
def train_model (model, train_text, train_emoji, test_text, test_emoji, learning_rate, epochs, batch_size):
    history = None

    return history

In [ ]:
history = train_model(model, train_en_vec, train_emoji_vec, test_en_vec, test_emoji_vec, 1e-4, 40, 256)

**Salida esperada**
```
Epoch 39/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0804 - loss: 0.1294 - val_accuracy: 0.0411 - val_loss: 4.8453
Epoch 40/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0805 - loss: 0.1232 - val_accuracy: 0.0409 - val_loss: 4.8631
```

veamos algunos ejemplos:

In [ ]:
current_token = 3
text_encoding = np.zeros([1,25])
text_encoding[0,0] = current_token

for i in range(24):
    token_prediction = model([np.expand_dims(test_en_vec[8], axis=0), text_encoding])
    current_token = np.argmax(token_prediction[0,i])
    text_encoding[0,i+1] = int(current_token)
    if current_token == 2:
        break

emoji_vocabulary = np.array(emoji_vectorizer.get_vocabulary())
en_vocabulary = np.array(en_vectorizer.get_vocabulary())
print(' '.join(en_vocabulary[list(test_en_vec[8].astype(int))]))
print(' '.join(emoji_vocabulary[list(text_encoding[0].astype(int))]))

## **Evaluar código**

In [ ]:
grader.run_test("Test 6_1", globals())

# **Evaluación**

In [ ]:
grader.submit_task(globals())

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://http://juselara.com/)
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*